# Raport z zadań z listy 2

## załadowanie zbiorów danych
Do wykonania zadań klasyfikacyjnych na ocenę 3.0 oraz na ocenę 3.5 używam zbioru Telco Customer Churn (Target Churn). Do zadań regresyjnych na ocenę 4.0, 4.5 oraz 5.0 używam zbioru Diamonds (target Price)

In [1]:
from file_operations import load_diamonds_data, load_telco_data, split_diamonds_data, split_telco_data
import pandas as pd

telco_df = load_telco_data()
telco_train_df, telco_test_df = split_telco_data(telco_df)

diamonds_df = load_diamonds_data()
diamonds_train_df, diamonds_test_df = split_diamonds_data(diamonds_df)

## Zadanie 3.0

In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
from preprocessing import prepare_telco_classification_data
from algorithms import TelcoDecisionTreeModel
import numpy as np

X_train, y_train, X_test, y_test = prepare_telco_classification_data(
    telco_train_df,
    telco_test_df,
)

model = TelcoDecisionTreeModel(max_depth=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

metrics_df = pd.DataFrame(
    {
        "metric": ["accuracy", "precision", "recall", "f1"],
        "value": [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred, pos_label="Yes"),
            recall_score(y_test, y_pred, pos_label="Yes"),
            f1_score(y_test, y_pred, pos_label="Yes"),
        ],
    }
)

display(metrics_df.round(4))

if hasattr(model, "model") and hasattr(model.model, "predict_proba"):
    y_test_binary = (y_test == "Yes").astype(int)
    y_scores = np.asarray(
        model.model.predict_proba(
            pd.get_dummies(X_test).reindex(columns=model.columns_, fill_value=0)
        )
    )[:, 1]
    print("Average precision:", round(average_precision_score(y_test_binary, y_scores), 4))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=["No", "Yes"]))


,metric,value
0,accuracy,0.7984
1,precision,0.6347
2,recall,0.5668
3,f1,0.5989


Average precision: 0.6223
Confusion matrix:
[[913 122]
 [162 212]]


Komentarz do wyników: Model drzewa decyzyjnego osiągnął accuracy na poziomie około 0.80, co jest wynikiem poprawnym, ale w tym zadaniu sama trafność nie wystarcza do rzetelnej oceny jakości modelu. Zbiór `Churn` jest niezbalansowany, dlatego większe znaczenie mają metryki precision, recall oraz F1 dla klasy `Yes`, czyli klientów rezygnujących z usług.

Precision na poziomie około 0.63 oznacza, że przewidywania odejścia klienta są dość często poprawne, ale model nadal generuje część fałszywych alarmów. Recall na poziomie około 0.57 pokazuje natomiast, że model wykrywa tylko część wszystkich rzeczywistych odejść, więc pomija istotną grupę klientów zagrożonych rezygnacją. F1 score na poziomie około 0.60 potwierdza, że model osiąga umiarkowaną równowagę pomiędzy precyzją i czułością.

Confusion matrix potwierdza, że model wyraźnie lepiej rozpoznaje klasę większościową `No` niż klasę mniejszościową `Yes`. Oznacza to, że model może stanowić sensowny punkt wyjścia do dalszej analizy, ale jego skuteczność w identyfikacji klientów odchodzących nadal jest ograniczona. 